# 🤖 Project 3.2 — Robot Joint Logger
**DSA for Mechatronics · Week 03**

> **Your task:** Run every cell from top to bottom.  
> At the end, a full report is printed automatically.  
> **Submit:** A screenshot or PDF of the final report cell output.  
> **Present:** Be ready to explain what each step does and why.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ║   This seeds your personal dataset — be honest!     ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

# ── seed everything from your ID ──────────────────────
import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready — run the next cells.")


## Step 1 — Generate your personal robot motion log

In [ ]:
# 3-DOF robot arm, randomly parameterised per student
N_STEPS  = random.randint(120, 200)      # recording length
N_JOINTS = 3
DT       = 0.1                           # seconds between steps
NOISE    = [round(random.uniform(1.5, 4.0), 2) for _ in range(N_JOINTS)]
LIMIT    = round(random.uniform(150.0, 250.0), 1)  # travel alarm threshold

motion_log = []
angles = [0.0] * N_JOINTS
for t in range(N_STEPS):
    deltas = [random.gauss(0, NOISE[j]) for j in range(N_JOINTS)]
    angles = [round(a + d, 2) for a, d in zip(angles, deltas)]
    motion_log.append(list(angles))

print("Motion log parameters:")
print(f"  Time steps  : {N_STEPS}  ({N_STEPS * DT:.1f} s at {1/DT:.0f} Hz)")
print(f"  Joints      : {N_JOINTS}")
print(f"  Noise (std) : J0={NOISE[0]}°  J1={NOISE[1]}°  J2={NOISE[2]}°")
print(f"  Alarm limit : {LIMIT}°")
print()
print(f"  {'t':>4}  {'Joint 0':>10}  {'Joint 1':>10}  {'Joint 2':>10}")
print(f"  {'─'*4}  {'─'*10}  {'─'*10}  {'─'*10}")
for t, row in enumerate(motion_log[:6]):
    print(f"  {t:4d}  {row[0]:>9.2f}°  {row[1]:>9.2f}°  {row[2]:>9.2f}°")
print("  ...")


## Step 2 — Extract per-joint time series (2D array access)

In [ ]:
def get_joint_series(log, joint_id):
    """Extract all angle values for one joint across all time steps."""
    return [row[joint_id] for row in log]

joints = [get_joint_series(motion_log, j) for j in range(N_JOINTS)]

print("Per-joint statistics:")
print(f"  {'Joint':>5}  {'Min':>8}  {'Max':>8}  {'Final':>8}  {'Range':>8}")
print(f"  {'─'*5}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}")
for j, series in enumerate(joints):
    print(f"  Joint {j}  {min(series):>7.2f}°  {max(series):>7.2f}°  "
          f"{series[-1]:>7.2f}°  {max(series)-min(series):>7.2f}°")


## Step 3 — Cumulative displacement with prefix sums

In [ ]:
def cumulative_displacement(angles):
    """
    Prefix sum of absolute step displacements.
    cum[t] = total angular distance travelled arriving at time step t.
    Range query: travel from t=a to t=b  =  cum[b] - cum[a]
    """
    cum = [0.0]
    for i in range(1, len(angles)):
        cum.append(cum[-1] + abs(angles[i] - angles[i - 1]))
    return cum

cums = [cumulative_displacement(joints[j]) for j in range(N_JOINTS)]

print("Cumulative angular displacement:")
print(f"  {'Joint':>5}  {'Total':>10}  {'0–25%':>10}  {'25–50%':>10}  {'50–75%':>10}  {'75–100%':>10}")
print(f"  {'─'*5}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}")
q = N_STEPS // 4
for j in range(N_JOINTS):
    c = cums[j]
    total = c[-1]
    q1 = c[q]   - c[0]
    q2 = c[2*q] - c[q]
    q3 = c[3*q] - c[2*q]
    q4 = c[-1]  - c[3*q]
    print(f"  Joint {j}  {total:>9.2f}°  {q1:>9.2f}°  {q2:>9.2f}°  {q3:>9.2f}°  {q4:>9.2f}°")


## Step 4 — Detect travel limit alarms

In [ ]:
def find_alarm(cum, limit):
    """Return first time step where cumulative travel exceeds limit, else None."""
    for t in range(len(cum)):
        if cum[t] > limit:
            return t
    return None

alarms = [find_alarm(cums[j], LIMIT) for j in range(N_JOINTS)]

print(f"Travel limit alarms (threshold = {LIMIT}°):")
print()
print(f"  {'Joint':>5}  {'Total travel':>13}  {'Alarm at':>10}  {'Time':>8}")
print(f"  {'─'*5}  {'─'*13}  {'─'*10}  {'─'*8}")
for j in range(N_JOINTS):
    total = cums[j][-1]
    alarm = alarms[j]
    if alarm is not None:
        print(f"  Joint {j}  {total:>12.2f}°  t = {alarm:03d}     {alarm*DT:>6.1f} s")
    else:
        print(f"  Joint {j}  {total:>12.2f}°  {'— never —':>10}")


## 📋 Final Report — this is what you submit

In [ ]:
most_active  = max(range(N_JOINTS), key=lambda j: cums[j][-1])
least_active = min(range(N_JOINTS), key=lambda j: cums[j][-1])
n_alarms     = sum(1 for a in alarms if a is not None)
status       = "✅ OK" if n_alarms == 0 else (f"⚠️  {n_alarms} ALARM(S)")

W = 56
print("╔" + "═"*W + "╗")
print("║" + " ROBOT ARM MOTION LOG — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<24}: {STUDENT_ID:<28}║")
print(f"║  {'Dataset seed':<24}: {_seed:<28}║")
print("╠" + "═"*W + "╣")
print("║  RECORDING PARAMETERS" + " "*(W-22) + "║")
print(f"║  {'Time steps':<24}: {N_STEPS:<5} ({N_STEPS*DT:.1f} s @ {1/DT:.0f} Hz){'':>12}║")
print(f"║  {'Noise std (J0/J1/J2)':<24}: {NOISE[0]}° / {NOISE[1]}° / {NOISE[2]}°{'':>19}║")
print(f"║  {'Alarm threshold':<24}: {LIMIT}°{'':>29}║")
print("╠" + "═"*W + "╣")
print("║  JOINT DISPLACEMENT" + " "*(W-20) + "║")
print(f"║  {'Joint':<6} {'Total travel':>14}  {'Alarm at':>10}  {'':>10}║")
print(f"║  {'─'*6} {'─'*14}  {'─'*10}  {'':>10}║")
for j in range(N_JOINTS):
    total = cums[j][-1]
    alarm = alarms[j]
    tag   = " ← MOST ACTIVE " if j == most_active else (" ← LEAST ACTIVE" if j == least_active else "               ")
    a_str = f"t={alarm:03d} ({alarm*DT:.1f}s)" if alarm is not None else "  never       "
    print(f"║  Joint {j}   {total:>12.2f}°  {a_str}  {tag}║")
print("╠" + "═"*W + "╣")
print(f"║  STATUS : {status:<{W-11}}║")
print("╚" + "═"*W + "╝")
